In [1]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv('~/Downloads/2042763_ejection_results.txt')

In [4]:
df.columns=['mass','time','particle_id','result']

In [5]:
N=[4500,4500,4500,4500,4500,4500,4500,4500,4500]

In [6]:
masses=df.mass.unique()


In [7]:

def compute_cumulative_outcome_fractions(df, masses, N):
    """
    df      : DataFrame with columns ['mass', 'time', 'particle_id', 'result']
              where result is 'Ejected' or 'Captured', and each row represents
              the time at which that particle's outcome occurred
    masses  : list/array of unique mass values, in the same order as N
    N       : list/array of total particle counts, one per mass in `masses`

    Returns
    -------
    ejection_df, capture_df : DataFrames indexed by mass, columns = sorted
                               unique time values, entries = CUMULATIVE
                               fraction of N with that result by that time
                               (i.e. fraction ejected/captured at or before
                               each time, monotonically non-decreasing
                               across each row)
    """
    if len(masses) != len(N):
        raise ValueError("masses and N must be the same length")

    mass_to_N = dict(zip(masses, N))

    times = np.sort(df["time"].unique())

    ejection_df = pd.DataFrame(index=masses, columns=times, dtype=float)
    capture_df = pd.DataFrame(index=masses, columns=times, dtype=float)

    for mass in masses:
        mass_df = df[df["mass"] == mass]
        denom = mass_to_N[mass]

        counts = (
            mass_df.groupby(["time", "result"])["particle_id"]
            .count()
            .unstack(fill_value=0)
        )

        # reindex to the full sorted time axis, filling any missing times
        # with 0 new outcomes at that time, then cumsum across time
        ejected_counts = counts.get("Ejected", pd.Series(0, index=counts.index))
        captured_counts = counts.get("Captured", pd.Series(0, index=counts.index))

        ejected_counts = ejected_counts.reindex(times, fill_value=0)
        captured_counts = captured_counts.reindex(times, fill_value=0)

        ejection_df.loc[mass] = (ejected_counts.cumsum() / denom).values
        capture_df.loc[mass] = (captured_counts.cumsum() / denom).values

    return ejection_df, capture_df

In [8]:
ejection_df, capture_df = compute_cumulative_outcome_fractions(df, masses, N)

In [10]:
"""
Convergence-regime diagnostics for monotonically increasing simulation traces.

Input
-----
A DataFrame `df` where:
    - each ROW is a simulation condition
    - each COLUMN is a time / step index, in increasing order
      (column labels should be numeric and sortable, e.g. step counts or times)
    - values in each row are monotonically increasing and (assumed) approaching
      an asymptote as columns -> large n

Output
------
A dict of DataFrames, all indexed the same way as `df` (same condition rows),
with columns aligned to the *later* of each pair/triple of time points used:

    'delta'   : increments  Δ_n = f(n) - f(n-1)
    'ratio'   : ratio of successive increments  r_n = Δ_(n+1) / Δ_n
                (the quantity your PI's bound depends on: r_n -> r* < 1
                 means geometric decay is a reasonable model; r_n creeping
                 toward 1 means algebraic/slower decay, and the naive
                 geometric-tail bound will be over-optimistic)
    'ratio_runmax' : running (trailing-window) max of r_n, i.e. the
                sup_{n>=N} r_n your PI's bound actually needs, not just the
                last observed r_n
    'aitken'  : Aitken Delta-squared extrapolated limit estimate L_n at each
                point, as a fast independent check on where the ratio-based
                bound is heading
    'local_power' : local estimate of exponent p_n assuming an algebraic tail
                f(n) - L ~ C / n^p, computed from three consecutive points
                without needing to know L in advance. If this p_n keeps
                drifting rather than settling, you're likely in the algebraic
                (or worse, logarithmic) regime.

Usage
-----
    diagnostics = compute_convergence_diagnostics(df)
    diagnostics['ratio']          # r_n vs time, per condition -- plot this first
    diagnostics['ratio_runmax']   # more honest version of the bound's r
    diagnostics['aitken']         # independent cross-check
"""



def compute_convergence_diagnostics(df: pd.DataFrame, ratio_window: int = 5) -> dict:
    """
    Parameters
    ----------
    df : pd.DataFrame
        Rows = conditions, columns = increasing time/step values, entries =
        monotonically increasing observed quantity f(n).
    ratio_window : int
        Trailing window size (in number of time points) used to compute the
        running max of r_n, i.e. an estimate of sup_{n>=N} r_n rather than
        just the single most recent ratio.

    Returns
    -------
    dict of pd.DataFrame : see module docstring.
    """
    # Ensure columns are sorted by time (numeric column labels assumed)
    df = df.reindex(sorted(df.columns), axis=1)
    cols = df.columns.to_numpy()

    if len(cols) < 4:
        raise ValueError(
            "Need at least 4 time points per row to compute ratio + local "
            "power diagnostics (2 lost to differencing, 1 more to ratio)."
        )

    # ---- increments: Delta_n = f(n) - f(n-1) --------------------------------
    delta = df.diff(axis=1)  # column n holds f(n) - f(n-1); first col is NaN
    delta = delta.iloc[:, 1:]  # drop the leading all-NaN column

    # ---- ratio of successive increments: r_n = Delta_(n+1) / Delta_n --------
    # Guard against division by ~0 (can happen if the trace plateaus exactly)
    delta_np = delta.to_numpy(dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio_np = delta_np[:, 1:] / delta_np[:, :-1]
    ratio_cols = delta.columns[1:]  # ratio at column n uses Delta_n, Delta_(n+1)
    ratio = pd.DataFrame(ratio_np, index=df.index, columns=ratio_cols)

    # ---- running (trailing-window) max of r_n --------------------------------
    # This approximates sup_{n>=N} r_n, which is what the geometric-tail bound
    # actually requires -- using only the last r_n is optimistic/noisy.
    ratio_runmax = ratio.T.rolling(window=ratio_window, min_periods=1).max().T

    # ---- Aitken Delta-squared extrapolated limit, L_n -----------------------
    # L_n = f(n) - [f(n+1)-f(n)]^2 / [f(n+2) - 2f(n+1) + f(n)]
    f = df.to_numpy(dtype=float)
    f0, f1, f2 = f[:, :-2], f[:, 1:-1], f[:, 2:]
    denom = f2 - 2 * f1 + f0
    with np.errstate(divide="ignore", invalid="ignore"):
        aitken_np = f0 - (f1 - f0) ** 2 / denom
    aitken_cols = df.columns[:-2]
    aitken = pd.DataFrame(aitken_np, index=df.index, columns=aitken_cols)

    # ---- local power-law exponent estimate, p_n ------------------------------
    # Assuming Delta_n ~ C * p / n^(p+1) locally (derivative of C/n^p form),
    # estimate p from three consecutive increments via a local log-log slope:
    #   p_n ≈ -log(Delta_(n+1)/Delta_n) / log(n_(n+1)/n_n)
    # using actual column (time) values, not just their integer order --
    # important if your time points aren't evenly spaced.
    n_vals = cols.astype(float)
    n1 = n_vals[1:-1]  # time values aligned with ratio's "later" increment
    n0 = n_vals[:-2]
    with np.errstate(divide="ignore", invalid="ignore"):
        log_ratio = np.log(ratio_np)
        log_n_ratio = np.log(n1 / n0)
        local_power_np = -log_ratio / log_n_ratio[np.newaxis, :]
    local_power = pd.DataFrame(local_power_np, index=df.index, columns=ratio_cols)

    return {
        "delta": delta,
        "ratio": ratio,
        "ratio_runmax": ratio_runmax,
        "aitken": aitken,
        "local_power": local_power,
    }




diagnostics = compute_convergence_diagnostics(ejection_df, ratio_window=5)

print("=== r_n (successive increment ratio) ===")
print(diagnostics["ratio"].round(3))
print("\n=== running max of r_n (window=5) ===")
print(diagnostics["ratio_runmax"].round(3))
print("\n=== local power-law exponent p_n ===")
print(diagnostics["local_power"].round(3))
print("\n=== Aitken extrapolated limit L_n ===")
print(diagnostics["aitken"].round(3))

=== r_n (successive increment ratio) ===
          2000.0     3000.0     4000.0     5000.0     6000.0     7000.0     \
0.000050        NaN        NaN        NaN        NaN        NaN        NaN   
0.000100        NaN        NaN        NaN        NaN        NaN        NaN   
0.000300        NaN        NaN        NaN        NaN        NaN        NaN   
0.000712        inf      0.000        inf      1.000      0.000        NaN   
0.001689        NaN        NaN        NaN        NaN        inf      1.400   
0.004004      0.000        inf     39.500      2.278      1.322      0.941   
0.009496     29.000     13.517      1.416      0.764      0.724      0.619   
0.030027     18.333      0.819      0.415      0.687      0.606      0.734   
0.100000      3.020      0.379      0.481      0.543      0.592      0.634   

          8000.0     9000.0     10000.0    11000.0    ...  4972000.0  \
0.000050        NaN        NaN        NaN        NaN  ...        0.0   
0.000100        NaN        NaN    

In [12]:
"""
Convergence-regime diagnostics for monotonically increasing simulation traces.

Input
-----
A DataFrame `df` where:
    - each ROW is a simulation condition
    - each COLUMN is a time / step index, in increasing order
      (column labels should be numeric and sortable, e.g. step counts or times)
    - values in each row are monotonically increasing and (assumed) approaching
      an asymptote as columns -> large n

Output
------
A dict of DataFrames, all indexed the same way as `df` (same condition rows),
with columns aligned to the *later* of each pair/triple of time points used:

    'delta'   : increments  Δ_n = f(n) - f(n-1)
    'ratio'   : ratio of successive increments  r_n = Δ_(n+1) / Δ_n
                (the quantity your PI's bound depends on: r_n -> r* < 1
                 means geometric decay is a reasonable model; r_n creeping
                 toward 1 means algebraic/slower decay, and the naive
                 geometric-tail bound will be over-optimistic)
    'ratio_runmax' : running (trailing-window) max of r_n, i.e. the
                sup_{n>=N} r_n your PI's bound actually needs, not just the
                last observed r_n
    'aitken'  : Aitken Delta-squared extrapolated limit estimate L_n at each
                point, as a fast independent check on where the ratio-based
                bound is heading
    'local_power' : local estimate of exponent p_n assuming an algebraic tail
                f(n) - L ~ C / n^p, computed from three consecutive points
                without needing to know L in advance. If this p_n keeps
                drifting rather than settling, you're likely in the algebraic
                (or worse, logarithmic) regime.

Usage
-----
    diagnostics = compute_convergence_diagnostics(df)
    diagnostics['ratio']          # r_n vs time, per condition -- plot this first
    diagnostics['ratio_runmax']   # more honest version of the bound's r
    diagnostics['aitken']         # independent cross-check
"""

import numpy as np
import pandas as pd


def compress_to_event_times(df: pd.DataFrame, min_delta: float = 0.0) -> pd.DataFrame:
    """
    Collapse each row down to just its event times: the columns where the
    value actually changed (Δ > min_delta) relative to the previous kept
    value, plus the first column as a starting point.

    This matters for discrete-event proportions, where f(n) is flat between
    events -- feeding those flat stretches into ratio/Aitken formulas causes
    0/0 (NaN) or x/0 (inf) since those formulas assume every step moves.

    Rows can have different numbers of events; short rows are right-padded
    with NaN so the result is still a rectangular DataFrame. The column
    labels of the output are meaningless as a shared time axis (each row's
    "column i" is that row's i-th event, not a common time point) -- keep
    track of true event times separately if you need them, e.g. by also
    compressing a parallel DataFrame of time-values, or by returning to
    per-row analysis for anything time-sensitive like elapsed-time-to-event.

    Parameters
    ----------
    df : pd.DataFrame
        Rows = conditions, columns = increasing time/step values.
    min_delta : float
        Minimum increment to count as a real event (use a small positive
        number instead of 0 if your data has floating-point noise on
        nominally-flat stretches).

    Returns
    -------
    pd.DataFrame, same index as df, columns = 0, 1, 2, ... (event index),
    NaN-padded on the right for rows with fewer events than the max.
    """
    df = df.reindex(sorted(df.columns), axis=1)
    compressed_rows = []
    for _, row in df.iterrows():
        vals = row.to_numpy(dtype=float)
        keep = [vals[0]]
        for v in vals[1:]:
            if v - keep[-1] > min_delta:
                keep.append(v)
        compressed_rows.append(keep)

    max_len = max(len(r) for r in compressed_rows)
    padded = [r + [np.nan] * (max_len - len(r)) for r in compressed_rows]
    return pd.DataFrame(padded, index=df.index, columns=range(max_len))


def compute_convergence_diagnostics(df: pd.DataFrame, ratio_window: int = 5) -> dict:
    """
    Parameters
    ----------
    df : pd.DataFrame
        Rows = conditions, columns = increasing time/step values, entries =
        monotonically increasing observed quantity f(n).
    ratio_window : int
        Trailing window size (in number of time points) used to compute the
        running max of r_n, i.e. an estimate of sup_{n>=N} r_n rather than
        just the single most recent ratio.

    Returns
    -------
    dict of pd.DataFrame : see module docstring.
    """
    # Ensure columns are sorted by time (numeric column labels assumed)
    df = df.reindex(sorted(df.columns), axis=1)
    cols = df.columns.to_numpy()

    if len(cols) < 4:
        raise ValueError(
            "Need at least 4 time points per row to compute ratio + local "
            "power diagnostics (2 lost to differencing, 1 more to ratio)."
        )

    # ---- increments: Delta_n = f(n) - f(n-1) --------------------------------
    delta = df.diff(axis=1)  # column n holds f(n) - f(n-1); first col is NaN
    delta = delta.iloc[:, 1:]  # drop the leading all-NaN column

    # ---- ratio of successive increments: r_n = Delta_(n+1) / Delta_n --------
    # Guard against division by ~0 (can happen if the trace plateaus exactly)
    delta_np = delta.to_numpy(dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio_np = delta_np[:, 1:] / delta_np[:, :-1]
    ratio_cols = delta.columns[1:]  # ratio at column n uses Delta_n, Delta_(n+1)
    ratio = pd.DataFrame(ratio_np, index=df.index, columns=ratio_cols)

    # ---- running (trailing-window) max of r_n --------------------------------
    # This approximates sup_{n>=N} r_n, which is what the geometric-tail bound
    # actually requires -- using only the last r_n is optimistic/noisy.
    ratio_runmax = ratio.T.rolling(window=ratio_window, min_periods=1).max().T

    # ---- Aitken Delta-squared extrapolated limit, L_n -----------------------
    # L_n = f(n) - [f(n+1)-f(n)]^2 / [f(n+2) - 2f(n+1) + f(n)]
    f = df.to_numpy(dtype=float)
    f0, f1, f2 = f[:, :-2], f[:, 1:-1], f[:, 2:]
    denom = f2 - 2 * f1 + f0
    with np.errstate(divide="ignore", invalid="ignore"):
        aitken_np = f0 - (f1 - f0) ** 2 / denom
    aitken_cols = df.columns[:-2]
    aitken = pd.DataFrame(aitken_np, index=df.index, columns=aitken_cols)

    # ---- local power-law exponent estimate, p_n ------------------------------
    # Assuming Delta_n ~ C * p / n^(p+1) locally (derivative of C/n^p form),
    # estimate p from three consecutive increments via a local log-log slope:
    #   p_n ≈ -log(Delta_(n+1)/Delta_n) / log(n_(n+1)/n_n)
    # using actual column (time) values, not just their integer order --
    # important if your time points aren't evenly spaced.
    n_vals = cols.astype(float)
    n1 = n_vals[1:-1]  # time values aligned with ratio's "later" increment
    n0 = n_vals[:-2]
    with np.errstate(divide="ignore", invalid="ignore"):
        log_ratio = np.log(ratio_np)
        log_n_ratio = np.log(n1 / n0)
        local_power_np = -log_ratio / log_n_ratio[np.newaxis, :]
    local_power = pd.DataFrame(local_power_np, index=df.index, columns=ratio_cols)

    return {
        "delta": delta,
        "ratio": ratio,
        "ratio_runmax": ratio_runmax,
        "aitken": aitken,
        "local_power": local_power,
    }


compressed = compress_to_event_times(ejection_df)

diagnostics = compute_convergence_diagnostics(compressed, ratio_window=5)

print("=== r_n (successive increment ratio) ===")
print(diagnostics["ratio"].round(3))
print("\n=== running max of r_n (window=5) ===")
print(diagnostics["ratio_runmax"].round(3))
print("\n=== local power-law exponent p_n ===")
print(diagnostics["local_power"].round(3))
print("\n=== Aitken extrapolated limit L_n ===")
print(diagnostics["aitken"].round(3))

# --- Demo with a discrete-event / plateau-heavy trace --------------------
# Simulates what a proportion-of-events-occurred trace looks like: flat
# for many steps, then jumps at event times, approaching an asymptote.



print("\n=== Compressed to event times (this is what removes the NaNs) ===")
print(compressed.iloc[:, :10])

event_diagnostics = compute_convergence_diagnostics(compressed, ratio_window=5)
print("\n=== r_n on compressed (event-indexed) sequence ===")
print(event_diagnostics["ratio"].round(3).iloc[:, -10:])

=== r_n (successive increment ratio) ===
            2       3      4      5      6      7      8      9      10    \
0.000050   1.000   1.000  1.000  1.000  1.000  1.000  1.000  1.000  1.000   
0.000100   1.000   1.000  1.000  1.000  1.000  1.000  2.000  0.500  1.000   
0.000300   1.000   1.000  1.000  1.000  2.000  1.000  1.000  0.500  1.000   
0.000712   1.000   1.000  1.000  3.000  1.000  0.667  2.000  1.750  1.286   
0.001689   1.400   3.857  2.037  0.964  1.472  1.064  1.301  0.852  1.054   
0.004004   2.000  39.500  2.278  1.322  0.941  1.067  0.749  0.994  0.742   
0.009496  29.000  13.517  1.416  0.764  0.724  0.619  0.642  0.787  0.844   
0.030027  18.333   0.819  0.415  0.687  0.606  0.734  0.850  0.647  0.841   
0.100000   3.020   0.379  0.481  0.543  0.592  0.634  0.756  1.176  0.425   

           11    ...  1685  1686  1687  1688  1689  1690  1691  1692  1693  \
0.000050  1.000  ...   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
0.000100  1.000  ...   1.0   1.0

In [ ]:
compute